# 🎮 Game Analytics Engine

> **Case real:** backend analítico de um jogo multiplayer online (MMO/Battle Royale style).  
> Cobre: **variáveis · funções · *args · **kwargs · decoradores · cache · exceptions · NumPy**

---

## Contexto do Projeto

Você é o engenheiro de sistemas responsável pelo **pipeline de dados em tempo real** de um jogo multiplayer.  
A cada partida, o servidor envia eventos de jogadores, kills, dano, sessões e métricas de performance.  
Seu sistema precisa:

- Registrar e validar eventos de jogadores
- Calcular métricas de performance (KDA, win rate, dano médio)
- Cachear resultados pesados para não sobrecarregar o banco
- Auditar e logar toda operação crítica com decoradores
- Rodar análises estatísticas com NumPy

---

## 📦 Imports

In [1]:
import time
import numpy as np
from datetime import datetime

---
## 1. Variáveis — Ficha de Jogador

Cada jogador conectado ao servidor tem uma ficha com seus dados de sessão atual.
Usamos variáveis de tipos diferentes para representar cada atributo.

In [2]:
# ── Ficha de jogador (dados de sessão ativa) ─────────────────────────────────

player_id       = "PLR-00421"        # str  — identificador único
username        = "ShadowByte_99"    # str  — nick no jogo
nivel           = 47                 # int  — nível atual
xp_atual        = 83_450             # int  — XP acumulado (underscore = legibilidade)
hp_percentual   = 0.68               # float — vida restante (0.0 a 1.0)
esta_vivo       = True               # bool — status na partida
regiao          = "SA-EAST-1"        # str  — servidor de origem
ping_ms         = 24                 # int  — latência em milissegundos

# Exibindo a ficha no console do servidor
print("=" * 40)
print("  FICHA DE JOGADOR — SESSÃO ATIVA")
print("=" * 40)
print(f"  ID       : {player_id}")
print(f"  Username : {username}")
print(f"  Nível    : {nivel}")
print(f"  XP       : {xp_atual:,}")           # :, → formata milhar
print(f"  HP       : {hp_percentual * 100:.0f}%")
print(f"  Vivo     : {esta_vivo}")
print(f"  Região   : {regiao}")
print(f"  Ping     : {ping_ms}ms")
print("=" * 40)

# Tipos de cada variável
print("\n[DEBUG] Tipos:")
for var, val in [("player_id", player_id), ("nivel", nivel),
                 ("hp_percentual", hp_percentual), ("esta_vivo", esta_vivo)]:
    print(f"  {var:15} → {type(val).__name__}")

  FICHA DE JOGADOR — SESSÃO ATIVA
  ID       : PLR-00421
  Username : ShadowByte_99
  Nível    : 47
  XP       : 83,450
  HP       : 68%
  Vivo     : True
  Região   : SA-EAST-1
  Ping     : 24ms

[DEBUG] Tipos:
  player_id       → str
  nivel           → int
  hp_percentual   → float
  esta_vivo       → bool


---
## 2. Funções — Cálculo de KDA e Win Rate

KDA (Kill/Death/Assist ratio) é a métrica mais usada em jogos competitivos.  
Funções puras e reutilizáveis são a base de qualquer sistema de analytics.

In [3]:
# ── Funções de métricas de performance ───────────────────────────────────────

def calcular_kda(kills: int, deaths: int, assists: int) -> float:
    """
    Calcula o KDA ratio de um jogador.
    KDA = (kills + assists) / max(1, deaths)  → evita divisão por zero
    """
    return round((kills + assists) / max(1, deaths), 2)


def calcular_winrate(vitorias: int, total_partidas: int) -> float:
    """
    Retorna o win rate em percentual (0.0 a 100.0).
    Retorna 0.0 se o jogador não tem partidas registradas.
    """
    if total_partidas == 0:
        return 0.0
    return round((vitorias / total_partidas) * 100, 1)


def classificar_jogador(kda: float, winrate: float) -> str:
    """
    Classifica o jogador com base no KDA e win rate combinados.
    Retorna a tier do jogador: Bronze, Silver, Gold, Platinum, Diamond.
    """
    score = (kda * 0.6) + (winrate / 100 * 10 * 0.4)   # score ponderado

    if score >= 8:   return "💎 Diamond"
    elif score >= 6: return "🥇 Platinum"
    elif score >= 4: return "⭐ Gold"
    elif score >= 2: return "🥈 Silver"
    else:            return "🥉 Bronze"


# Testando com 3 perfis de jogadores
jogadores_teste = [
    ("ShadowByte_99", 340, 87, 210, 178, 220),   # (nick, k, d, a, wins, total)
    ("NoviceGamer01",  45, 180, 30,  12, 100),
    ("ProSniper_X",   520, 60, 180, 290, 330),
]

print(f"{'JOGADOR':<18} {'KDA':>6} {'WINRATE':>9} {'TIER'}")
print("-" * 48)
for nick, k, d, a, w, t in jogadores_teste:
    kda  = calcular_kda(k, d, a)
    wr   = calcular_winrate(w, t)
    tier = classificar_jogador(kda, wr)
    print(f"{nick:<18} {kda:>6.2f} {wr:>8.1f}%  {tier}")

JOGADOR               KDA   WINRATE TIER
------------------------------------------------
ShadowByte_99        6.32     80.9%  🥇 Platinum
NoviceGamer01        0.42     12.0%  🥉 Bronze
ProSniper_X         11.67     87.9%  💎 Diamond


---
## 3. `*args` e Tuplas — Processamento de Eventos de Partida

Durante uma partida, o servidor recebe uma quantidade variável de eventos de dano.  
`*args` é perfeito para funções que precisam processar N entradas sem saber o total antecipadamente.

In [4]:
# ── Processador de eventos de dano ───────────────────────────────────────────

def processar_dano(player_id: str, *eventos_dano):
    """
    Processa todos os eventos de dano causados por um jogador em uma partida.
    Cada evento é um int representando dano causado em um hit.
    """
    print(f"\n[MATCH EVENT] Player: {player_id}")
    print(f"  Hits registrados : {len(eventos_dano)}")
    print(f"  Dados brutos     : {eventos_dano}")          # → tupla imutável!

    dano_total  = sum(eventos_dano)
    dano_medio  = dano_total / len(eventos_dano)
    dano_max    = max(eventos_dano)
    headshots   = sum(1 for d in eventos_dano if d >= 100)  # dano >= 100 = headshot

    print(f"  Dano total       : {dano_total}")
    print(f"  Dano médio/hit   : {dano_medio:.1f}")
    print(f"  Maior hit        : {dano_max}")
    print(f"  Headshots        : {headshots}")

    return {
        "player": player_id,
        "total":  dano_total,
        "medio":  round(dano_medio, 1),
        "max":    dano_max,
        "hs":     headshots
    }


# Simulando hits de diferentes jogadores na mesma partida
r1 = processar_dano("PLR-00421", 45, 120, 80, 33, 150, 95, 110, 62, 88)
r2 = processar_dano("PLR-00789", 30, 75, 140, 55)                         # menos hits
r3 = processar_dano("PLR-00112", 200, 180, 175, 190, 210, 165, 195, 188, 201, 177, 185, 193)  # sniper


[MATCH EVENT] Player: PLR-00421
  Hits registrados : 9
  Dados brutos     : (45, 120, 80, 33, 150, 95, 110, 62, 88)
  Dano total       : 783
  Dano médio/hit   : 87.0
  Maior hit        : 150
  Headshots        : 3

[MATCH EVENT] Player: PLR-00789
  Hits registrados : 4
  Dados brutos     : (30, 75, 140, 55)
  Dano total       : 300
  Dano médio/hit   : 75.0
  Maior hit        : 140
  Headshots        : 1

[MATCH EVENT] Player: PLR-00112
  Hits registrados : 12
  Dados brutos     : (200, 180, 175, 190, 210, 165, 195, 188, 201, 177, 185, 193)
  Dano total       : 2259
  Dano médio/hit   : 188.2
  Maior hit        : 210
  Headshots        : 12


---
## 4. `**kwargs` e Dicionários — Registro de Partida

Cada tipo de partida (ranked, casual, torneio) tem atributos diferentes.  
`**kwargs` permite criar um sistema de registro flexível sem quebrar a interface quando novos modos de jogo são adicionados.

In [5]:
# ── Registro flexível de partidas ────────────────────────────────────────────

match_registry = []   # banco em memória (simulado)


def registrar_partida(match_id: str, modo: str, **config):
    """
    Registra uma partida com configurações variáveis por modo de jogo.
    **config absorve qualquer campo extra sem quebrar o sistema.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    partida = {
        "match_id":  match_id,
        "modo":      modo,
        "timestamp": timestamp,
        **config          # ← desempacota kwargs direto no dict!
    }

    match_registry.append(partida)

    print(f"\n[MATCH REGISTERED] {match_id} — {modo.upper()}")
    for chave, valor in partida.items():
        print(f"  {chave:<20} : {valor}")

    return partida


# Cada modo de jogo tem campos completamente diferentes — kwargs lida com tudo
registrar_partida(
    "MATCH-9981", "ranked",
    mapa="Distrito_Neon",
    jogadores=10,
    duracao_seg=1847,
    mvp="ShadowByte_99",
    temporada="S7"
)

registrar_partida(
    "MATCH-9982", "torneio",
    mapa="Bunker_Alpha",
    times=2,
    prize_pool_usd=5000,
    transmissao_ao_vivo=True,
    organizador="ESL_Brazil"
)

registrar_partida(
    "MATCH-9983", "casual",
    mapa="Praia_Caos",
    jogadores=20,
    modo_especial="gravidade_reduzida"
)

print(f"\n[REGISTRY] Total de partidas registradas: {len(match_registry)}")


[MATCH REGISTERED] MATCH-9981 — RANKED
  match_id             : MATCH-9981
  modo                 : ranked
  timestamp            : 2026-06-29 09:56:41
  mapa                 : Distrito_Neon
  jogadores            : 10
  duracao_seg          : 1847
  mvp                  : ShadowByte_99
  temporada            : S7

[MATCH REGISTERED] MATCH-9982 — TORNEIO
  match_id             : MATCH-9982
  modo                 : torneio
  timestamp            : 2026-06-29 09:56:41
  mapa                 : Bunker_Alpha
  times                : 2
  prize_pool_usd       : 5000
  transmissao_ao_vivo  : True
  organizador          : ESL_Brazil

[MATCH REGISTERED] MATCH-9983 — CASUAL
  match_id             : MATCH-9983
  modo                 : casual
  timestamp            : 2026-06-29 09:56:41
  mapa                 : Praia_Caos
  jogadores            : 20
  modo_especial        : gravidade_reduzida

[REGISTRY] Total de partidas registradas: 3


---
## 5. Decoradores — Auditoria e Log de Operações Críticas

Em sistemas de jogos online, toda operação que envolve ranking, ban ou transação financeira  
precisa ser auditada. Decoradores são a solução elegante: o código de negócio fica limpo,  
a infraestrutura de log fica separada.

In [6]:
# ── Decoradores de infraestrutura ─────────────────────────────────────────────

import time

def audit_log(func):
    """Registra quem chamou a função, quando e o resultado."""
    def wrapper(*args, **kwargs):
        ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        print(f"[AUDIT {ts}] ▶ {func.__name__}() | args={args} kwargs={kwargs}")
        resultado = func(*args, **kwargs)
        print(f"[AUDIT {ts}] ✓ {func.__name__}() → {resultado}")
        return resultado
    return wrapper


def medir_latencia(func):
    """Mede o tempo de execução — essencial para SLAs de servidor de jogo."""
    def wrapper(*args, **kwargs):
        inicio    = time.perf_counter()
        resultado = func(*args, **kwargs)
        latencia  = (time.perf_counter() - inicio) * 1000
        status    = "✅ OK" if latencia < 100 else "⚠️  SLOW"
        print(f"[PERF] {func.__name__}() → {latencia:.2f}ms {status}")
        return resultado
    return wrapper


# Aplicando os dois decoradores em operações críticas
# A ordem importa: audit_log é o mais externo (executado primeiro)

@audit_log
@medir_latencia
def aplicar_ban(player_id: str, motivo: str, duracao_horas: int = 24):
    """Aplica ban em um jogador (operação crítica — precisa de audit trail)."""
    time.sleep(0.002)   # simula operação no banco
    return {"status": "BANNED", "player": player_id, "horas": duracao_horas}


@audit_log
@medir_latencia
def atualizar_ranking(player_id: str, pontos_delta: int):
    """Atualiza os pontos de ranking de um jogador após uma partida."""
    time.sleep(0.001)   # simula escrita no Redis
    novo_mmr = 1850 + pontos_delta   # MMR simulado
    return {"player": player_id, "novo_mmr": novo_mmr}


print("=" * 55)
print(" SIMULANDO OPERAÇÕES PÓS-PARTIDA")
print("=" * 55)

atualizar_ranking("PLR-00421", +25)
print()
atualizar_ranking("PLR-00789", -18)
print()
aplicar_ban("PLR-09999", motivo="cheating_aimbot", duracao_horas=720)

 SIMULANDO OPERAÇÕES PÓS-PARTIDA
[AUDIT 09:56:41.657] ▶ wrapper() | args=('PLR-00421', 25) kwargs={}
[PERF] atualizar_ranking() → 1.11ms ✅ OK
[AUDIT 09:56:41.657] ✓ wrapper() → {'player': 'PLR-00421', 'novo_mmr': 1875}

[AUDIT 09:56:41.658] ▶ wrapper() | args=('PLR-00789', -18) kwargs={}
[PERF] atualizar_ranking() → 1.08ms ✅ OK
[AUDIT 09:56:41.658] ✓ wrapper() → {'player': 'PLR-00789', 'novo_mmr': 1832}

[AUDIT 09:56:41.660] ▶ wrapper() | args=('PLR-09999',) kwargs={'motivo': 'cheating_aimbot', 'duracao_horas': 720}
[PERF] aplicar_ban() → 2.10ms ✅ OK
[AUDIT 09:56:41.660] ✓ wrapper() → {'status': 'BANNED', 'player': 'PLR-09999', 'horas': 720}


{'status': 'BANNED', 'player': 'PLR-09999', 'horas': 720}

---
## 6. Cache — Player Profile Service

Consultas de perfil acontecem centenas de vezes por segundo em jogos populares.  
Sem cache, o banco de dados colapsaria. O cache aqui simula o padrão **Cache-Aside**,  
o mais usado em sistemas de jogos com Redis.

In [7]:
# ── Cache-Aside Pattern — Player Profile Service ─────────────────────────────

# Simulação do banco de dados (leitura lenta)
_database = {
    "PLR-00421": {"username": "ShadowByte_99",  "nivel": 47, "rank": "Platinum", "pais": "BR"},
    "PLR-00789": {"username": "DesertEagle_42",  "nivel": 31, "rank": "Gold",     "pais": "AR"},
    "PLR-00112": {"username": "ProSniper_X",     "nivel": 82, "rank": "Diamond",  "pais": "BR"},
    "PLR-00055": {"username": "CasualPlayer_77", "nivel": 12, "rank": "Bronze",   "pais": "CL"},
    "PLR-00300": {"username": "TheCommander",    "nivel": 61, "rank": "Platinum", "pais": "BR"},
}

_cache_perfis   = {}     # cache em memória (simula Redis)
_access_counter = {}     # contador de acessos por player
_cache_hits     = 0
_db_hits        = 0


def buscar_perfil(player_id: str) -> dict | None:
    """Busca perfil com Cache-Aside: verifica cache antes de ir ao banco."""
    global _cache_hits, _db_hits

    # Incrementa contador de acesso
    _access_counter[player_id] = _access_counter.get(player_id, 0) + 1

    # Cache HIT
    if player_id in _cache_perfis:
        _cache_hits += 1
        print(f"  [CACHE HIT]  {player_id} → {_cache_perfis[player_id]['username']}")
        return _cache_perfis[player_id]

    # Cache MISS → vai ao banco
    _db_hits += 1
    time.sleep(0.05)   # latência simulada de DB query
    perfil = _database.get(player_id)

    if perfil:
        _cache_perfis[player_id] = perfil   # popula o cache
        print(f"  [DB QUERY]   {player_id} → {perfil['username']} (cacheado)")
    else:
        print(f"  [NOT FOUND]  {player_id}")

    return perfil


# Simulando requisições de um lobby de 10 jogadores
# (jogadores consultam perfis uns dos outros antes da partida)
requisicoes = [
    "PLR-00421", "PLR-00789", "PLR-00421",   # ShadowByte consultado 2x
    "PLR-00112", "PLR-00055", "PLR-00421",   # ShadowByte consultado 3x
    "PLR-00300", "PLR-00789", "PLR-00999",   # PLR-999 não existe
    "PLR-00112", "PLR-00300", "PLR-00112",
]

print("=" * 50)
print(" LOBBY — REQUISIÇÕES DE PERFIL")
print("=" * 50)

for pid in requisicoes:
    buscar_perfil(pid)

total_req = len(requisicoes)
print(f"\n{'=':=<50}")
print(f"  Total requisições : {total_req}")
print(f"  DB queries        : {_db_hits}  ({_db_hits/total_req*100:.0f}%)")
print(f"  Cache hits        : {_cache_hits}  ({_cache_hits/total_req*100:.0f}%)")
print(f"\n  Acessos por jogador:")
for pid, count in sorted(_access_counter.items(), key=lambda x: -x[1]):
    username = _database.get(pid, {}).get('username', 'desconhecido')
    print(f"    {username:<22} : {count}x")

 LOBBY — REQUISIÇÕES DE PERFIL
  [DB QUERY]   PLR-00421 → ShadowByte_99 (cacheado)
  [DB QUERY]   PLR-00789 → DesertEagle_42 (cacheado)
  [CACHE HIT]  PLR-00421 → ShadowByte_99
  [DB QUERY]   PLR-00112 → ProSniper_X (cacheado)
  [DB QUERY]   PLR-00055 → CasualPlayer_77 (cacheado)
  [CACHE HIT]  PLR-00421 → ShadowByte_99
  [DB QUERY]   PLR-00300 → TheCommander (cacheado)
  [CACHE HIT]  PLR-00789 → DesertEagle_42
  [NOT FOUND]  PLR-00999
  [CACHE HIT]  PLR-00112 → ProSniper_X
  [CACHE HIT]  PLR-00300 → TheCommander
  [CACHE HIT]  PLR-00112 → ProSniper_X

  Total requisições : 12
  DB queries        : 6  (50%)
  Cache hits        : 6  (50%)

  Acessos por jogador:
    ShadowByte_99          : 3x
    ProSniper_X            : 3x
    DesertEagle_42         : 2x
    TheCommander           : 2x
    CasualPlayer_77        : 1x
    desconhecido           : 1x


---
## 7. Exceptions — Validação de Transações de In-Game Store

A loja do jogo lida com dinheiro real (ou moeda virtual).  
Erros silenciosos aqui = prejuízo real. Exceptions bem definidas garantem que
nenhum erro passe despercebido, com mensagens claras para logs e para o jogador.

In [8]:
# ── Sistema de transações da In-Game Store ────────────────────────────────────

# Exceptions customizadas — cada erro tem identidade própria
class SaldoInsuficienteError(Exception):    pass
class ItemNaoEncontradoError(Exception):    pass
class JogadorBanidoError(Exception):        pass
class LimiteTransacaoDiariaError(Exception):pass


# Catálogo simulado da loja
_catalogo = {
    "skin_dragon":    {"nome": "Skin Dragão Ancestral",  "preco": 1500},
    "passe_batalha":  {"nome": "Passe de Batalha S7",    "preco": 950},
    "emote_dance":    {"nome": "Emote: Victory Dance",   "preco": 300},
    "arma_gold":      {"nome": "Skin Arma Gold Edition", "preco": 2200},
}

_jogadores_banidos = {"PLR-09999"}


def comprar_item(player_id: str, item_id: str, saldo_gc: int,
                 gasto_hoje: int = 0, limite_diario: int = 5000) -> dict:
    """
    Processa uma compra na In-Game Store com validações completas.
    saldo_gc = saldo em GameCoins (moeda do jogo)
    """
    try:
        # Validação 1: jogador banido
        if player_id in _jogadores_banidos:
            raise JogadorBanidoError(
                f"Player {player_id} está banido e não pode realizar compras.")

        # Validação 2: item existe
        if item_id not in _catalogo:
            raise ItemNaoEncontradoError(
                f"Item '{item_id}' não encontrado no catálogo.")

        item  = _catalogo[item_id]
        preco = item["preco"]

        # Validação 3: limite diário
        if gasto_hoje + preco > limite_diario:
            raise LimiteTransacaoDiariaError(
                f"Limite diário de {limite_diario} GC atingido. "
                f"Gasto hoje: {gasto_hoje} GC | Tentativa: {preco} GC")

        # Validação 4: saldo suficiente
        if saldo_gc < preco:
            raise SaldoInsuficienteError(
                f"Saldo insuficiente: {saldo_gc} GC disponível, {preco} GC necessário.")

        # Compra aprovada
        novo_saldo = saldo_gc - preco
        resultado  = {
            "status":     "SUCCESS",
            "player":     player_id,
            "item":       item["nome"],
            "pago":       preco,
            "novo_saldo": novo_saldo
        }
        print(f"  ✅ COMPRA APROVADA: {item['nome']} por {preco} GC | Saldo: {novo_saldo} GC")
        return resultado

    except JogadorBanidoError        as e: print(f"  🚫 BANIDO          : {e}")
    except ItemNaoEncontradoError    as e: print(f"  ❌ ITEM INVÁLIDO   : {e}")
    except LimiteTransacaoDiariaError as e: print(f"  ⛔ LIMITE DIÁRIO  : {e}")
    except SaldoInsuficienteError    as e: print(f"  💸 SALDO INSUF.   : {e}")
    finally:
        print(f"  [LOG] Tentativa de compra registrada — player={player_id}, item={item_id}")


# Cenários de teste
print("=" * 60)
print(" IN-GAME STORE — PROCESSANDO TRANSAÇÕES")
print("=" * 60)

print("\n[TX-001] Compra válida:")
comprar_item("PLR-00421", "passe_batalha", saldo_gc=2000)

print("\n[TX-002] Saldo insuficiente:")
comprar_item("PLR-00789", "arma_gold", saldo_gc=500)

print("\n[TX-003] Item não existe:")
comprar_item("PLR-00112", "item_inexistente_xp", saldo_gc=9999)

print("\n[TX-004] Jogador banido tentando comprar:")
comprar_item("PLR-09999", "emote_dance", saldo_gc=5000)

print("\n[TX-005] Limite diário atingido:")
comprar_item("PLR-00300", "skin_dragon", saldo_gc=9999, gasto_hoje=4800)

 IN-GAME STORE — PROCESSANDO TRANSAÇÕES

[TX-001] Compra válida:
  ✅ COMPRA APROVADA: Passe de Batalha S7 por 950 GC | Saldo: 1050 GC
  [LOG] Tentativa de compra registrada — player=PLR-00421, item=passe_batalha

[TX-002] Saldo insuficiente:
  💸 SALDO INSUF.   : Saldo insuficiente: 500 GC disponível, 2200 GC necessário.
  [LOG] Tentativa de compra registrada — player=PLR-00789, item=arma_gold

[TX-003] Item não existe:
  ❌ ITEM INVÁLIDO   : Item 'item_inexistente_xp' não encontrado no catálogo.
  [LOG] Tentativa de compra registrada — player=PLR-00112, item=item_inexistente_xp

[TX-004] Jogador banido tentando comprar:
  🚫 BANIDO          : Player PLR-09999 está banido e não pode realizar compras.
  [LOG] Tentativa de compra registrada — player=PLR-09999, item=emote_dance

[TX-005] Limite diário atingido:
  ⛔ LIMITE DIÁRIO  : Limite diário de 5000 GC atingido. Gasto hoje: 4800 GC | Tentativa: 1500 GC
  [LOG] Tentativa de compra registrada — player=PLR-00300, item=skin_dragon


---
## 8. NumPy — Match Analytics Dashboard

Ao fim de uma temporada, o sistema precisa gerar relatórios completos de performance.  
NumPy permite processar milhares de partidas em milissegundos — a mesma base usada em  
engines de recomendação de matchmaking.

In [9]:
# ── Análise estatística de temporada com NumPy ────────────────────────────────

import numpy as np

# Dados de 30 partidas da temporada do ShadowByte_99
kills_por_partida  = np.array([8, 12, 3, 15, 7, 20, 5, 11, 9, 18,
                                6, 14, 2, 16, 10, 13, 4, 19, 8, 11,
                               17, 6, 12, 3, 14, 9, 21, 7, 15, 10])

dano_por_partida   = np.array([820, 1240, 310, 1580, 740, 2100, 490, 1120, 940, 1870,
                                630, 1450, 210, 1690, 1050, 1380, 420, 1950, 830, 1180,
                               1780, 610, 1290, 340, 1470, 980, 2250, 710, 1560, 1080])

resultado_partida  = np.array([1, 1, 0, 1, 0, 1, 0, 1, 1, 1,
                                0, 1, 0, 1, 1, 1, 0, 1, 0, 0,
                                1, 0, 1, 0, 1, 1, 1, 0, 1, 1])  # 1=vitória, 0=derrota

print("=" * 55)
print("  📊 SEASON REPORT — ShadowByte_99")
print("=" * 55)

# KPIs gerais
print("\n[KILLS]")
print(f"  Total    : {kills_por_partida.sum()}")
print(f"  Média    : {kills_por_partida.mean():.1f} por partida")
print(f"  Melhor   : {kills_por_partida.max()} kills")
print(f"  Pior     : {kills_por_partida.min()} kills")
print(f"  Desvio   : {kills_por_partida.std():.1f}  (consistência)")

print("\n[DANO]")
print(f"  Total    : {dano_por_partida.sum():,} de dano na temporada")
print(f"  Média    : {dano_por_partida.mean():.0f} por partida")
print(f"  Pico     : {dano_por_partida.max():,}")

print("\n[WIN RATE]")
total_vitorias = resultado_partida.sum()
total_partidas = len(resultado_partida)
winrate = (total_vitorias / total_partidas) * 100
print(f"  Vitórias : {total_vitorias}/{total_partidas}")
print(f"  Win Rate : {winrate:.1f}%")

# Análise condicional vetorizada — o poder do NumPy
print("\n[ANÁLISE AVANÇADA]")
partidas_carry   = kills_por_partida[kills_por_partida >= 15]
partidas_ruim    = kills_por_partida[kills_por_partida <= 5]
partidas_vitoria_kills = kills_por_partida[resultado_partida == 1]
partidas_derrota_kills = kills_por_partida[resultado_partida == 0]

print(f"  Partidas carry (≥15 kills)  : {len(partidas_carry)} partidas")
print(f"  Partidas ruins (≤5 kills)   : {len(partidas_ruim)} partidas")
print(f"  Média kills em vitórias     : {partidas_vitoria_kills.mean():.1f}")
print(f"  Média kills em derrotas     : {partidas_derrota_kills.mean():.1f}")

# Percentis para matchmaking
print("\n[MATCHMAKING STATS — PERCENTIS DE DANO]")
for p in [25, 50, 75, 90]:
    val = np.percentile(dano_por_partida, p)
    print(f"  P{p:<3} : {val:>7.0f} de dano")

# Correlação kills × dano
correlacao = np.corrcoef(kills_por_partida, dano_por_partida)[0, 1]
print(f"\n  Correlação kills↔dano : {correlacao:.3f} (esperado: próximo de 1.0)")

# Avaliação final
print("\n[AVALIAÇÃO FINAL]")
if winrate >= 60 and kills_por_partida.mean() >= 10:
    print("  🏆 Performance ELITE — Elegível para ranking Diamond")
elif winrate >= 50:
    print("  ⭐ Performance ACIMA DA MÉDIA — Mantendo Platinum")
else:
    print("  📉 Performance ABAIXO DA META — Risco de rebaixamento")

  📊 SEASON REPORT — ShadowByte_99

[KILLS]
  Total    : 325
  Média    : 10.8 por partida
  Melhor   : 21 kills
  Pior     : 2 kills
  Desvio   : 5.2  (consistência)

[DANO]
  Total    : 34,070 de dano na temporada
  Média    : 1136 por partida
  Pico     : 2,250

[WIN RATE]
  Vitórias : 19/30
  Win Rate : 63.3%

[ANÁLISE AVANÇADA]
  Partidas carry (≥15 kills)  : 8 partidas
  Partidas ruins (≤5 kills)   : 5 partidas
  Média kills em vitórias     : 13.8
  Média kills em derrotas     : 5.6

[MATCHMAKING STATS — PERCENTIS DE DANO]
  P25  :     718 de dano
  P50  :    1100 de dano
  P75  :    1538 de dano
  P90  :    1878 de dano

  Correlação kills↔dano : 0.999 (esperado: próximo de 1.0)

[AVALIAÇÃO FINAL]
  🏆 Performance ELITE — Elegível para ranking Diamond


---
## 9. Pipeline Completo — Fim de Partida

Integrando todos os módulos em um fluxo realista de pós-partida:
o que acontece nos servidores nos segundos após o `MATCH OVER`.

In [10]:
# ── Pipeline de pós-partida integrando todos os módulos ──────────────────────

def pipeline_pos_partida(match_id: str, resultados: list[dict]):
    """
    Executa o pipeline completo de pós-partida.
    resultados: lista de dicts com dados de cada jogador.
    """
    print(f"\n{'='*60}")
    print(f"  🏁 MATCH OVER — {match_id}")
    print(f"{'='*60}")

    for p in resultados:
        print(f"\n  ── {p['username']} ──")

        # Métricas (funções)
        kda  = calcular_kda(p['kills'], p['deaths'], p['assists'])
        tier = classificar_jogador(kda, p['winrate_historico'])

        # Dano (args)
        stats_dano = processar_dano(p['player_id'], *p['hits'])

        # Ranking (decorado com audit + latência)
        delta = +20 if p['venceu'] else -15
        atualizar_ranking(p['player_id'], delta)

        print(f"  Tier calculada : {tier}")


# Dados de uma partida finalizada
resultado_partida_final = [
    {
        "player_id": "PLR-00421", "username": "ShadowByte_99",
        "kills": 14, "deaths": 3, "assists": 6,
        "hits": [95, 110, 145, 80, 200, 75, 130, 88],
        "winrate_historico": 64.2, "venceu": True
    },
    {
        "player_id": "PLR-00789", "username": "DesertEagle_42",
        "kills": 5, "deaths": 9, "assists": 2,
        "hits": [40, 55, 30, 70, 48],
        "winrate_historico": 41.0, "venceu": False
    },
]

pipeline_pos_partida("MATCH-9981", resultado_partida_final)

print(f"\n\n{'='*60}")
print("  ✅ Pipeline concluído. Dados gravados.")
print(f"{'='*60}")


  🏁 MATCH OVER — MATCH-9981

  ── ShadowByte_99 ──

[MATCH EVENT] Player: PLR-00421
  Hits registrados : 8
  Dados brutos     : (95, 110, 145, 80, 200, 75, 130, 88)
  Dano total       : 923
  Dano médio/hit   : 115.4
  Maior hit        : 200
  Headshots        : 4
[AUDIT 09:56:42.225] ▶ wrapper() | args=('PLR-00421', 20) kwargs={}
[PERF] atualizar_ranking() → 1.08ms ✅ OK
[AUDIT 09:56:42.225] ✓ wrapper() → {'player': 'PLR-00421', 'novo_mmr': 1870}
  Tier calculada : 🥇 Platinum

  ── DesertEagle_42 ──

[MATCH EVENT] Player: PLR-00789
  Hits registrados : 5
  Dados brutos     : (40, 55, 30, 70, 48)
  Dano total       : 243
  Dano médio/hit   : 48.6
  Maior hit        : 70
  Headshots        : 0
[AUDIT 09:56:42.226] ▶ wrapper() | args=('PLR-00789', -15) kwargs={}
[PERF] atualizar_ranking() → 1.56ms ✅ OK
[AUDIT 09:56:42.226] ✓ wrapper() → {'player': 'PLR-00789', 'novo_mmr': 1835}
  Tier calculada : 🥈 Silver


  ✅ Pipeline concluído. Dados gravados.
